# Урок 18 — Від ручного контролю до автоматичного
## `@property` · Дескриптори · Архітектура валідації

---

> *«У Уроці 17 ти навчився контролювати дані вручну.  
> Сьогодні ти навчишся змусити Python робити це за тебе.»*

---

## Зміст

| # | Розділ |
|---|--------|
| 1 | Рекап Уроку 17 — «ми майже захистили об'єкт» |
| 2 | Проблема — `set_mpg()` можна обійти |
| 3 | Рішення 1 — `@property` |
| 4 | Обмеження `@property` — дублювання |
| 5 | Рішення 2 — Дескриптор `PositiveNumber` |
| 6 | Пояснення — що таке дескриптор зсередини |
| 7 | Порівняльна таблиця підходів |
| 8 | Завдання |
| 9 | Висновок |

---

## Розділ 1 — Рекап Уроку 17

У минулому уроці ми будували клас `Car` з:  
- `_mpg` — protected-атрибут  
- `set_mpg()` — метод з валідацією  
- `efficiency()` — поліморфний розрахунок  
- дандер-методами `__str__`, `__lt__`, `__eq__`

Швидко відновимо цю систему.

In [ ]:
import seaborn as sns
import pandas as pd

df = sns.load_dataset("mpg").dropna()

# --- Клас Car з Уроку 17 (без змін) ---
class Car:
    def __init__(self, name, mpg, weight, origin):
        self.name     = name
        self._mpg     = mpg       # protected: домовленість не чіпати напряму
        self._weight  = weight
        self.origin   = origin

    def set_mpg(self, value):
        # Захист: перевіряємо значення перед збереженням
        if value <= 0:
            print(f"set_mpg: {value} — недійсне значення, mpg не змінено")
            return
        self._mpg = value

    def efficiency(self):
        return round(self._mpg, 2)

    def __str__(self):
        return f"{self.name} — {self._mpg} mpg [{self.origin}]"

    def __repr__(self):
        return f"Car({self.name!r}, mpg={self._mpg})"

    def __lt__(self, other):
        return self._mpg < other._mpg

    def __eq__(self, other):
        if not isinstance(other, Car): return NotImplemented
        return self._mpg == other._mpg


class GasolineCar(Car):
    def efficiency(self): return round(self._mpg * 1.0, 2)

class DieselCar(Car):
    def efficiency(self): return round(self._mpg * 1.2, 2)

class ElectricCar(Car):
    def efficiency(self): return round(self._mpg * 2.5, 2)


# Перевіряємо, що система працює
car = GasolineCar("Ford Pinto", 18.0, 2601, "usa")
car.set_mpg(-50)        # Спрацьовує захист
print(f"mpg залишився: {car._mpg}")
car.set_mpg(25.0)       # Правильне значення
print(f"mpg оновлено:  {car._mpg}")

---

## ❌ Розділ 2 — Проблема: `set_mpg()` можна обійти

Здається, що ми все зробили правильно. `set_mpg(-50)` — захищено.  
Але є три реальних ризики:

1. **Користувач може обійти захист** — присвоїти `_mpg` напряму  
2. **Розробник може забути** — написати `self._mpg = value` замість `set_mpg()`  
3. **Не Pythonic** — `car.set_mpg(25)` vs `car.mpg = 25` (Java-стиль у Python-коді)

Подивимось на реальний злам:

In [ ]:
car = GasolineCar("Ford Pinto", 18.0, 2601, "usa")

# set_mpg захищає — але хто знає про нього?
car._mpg = -100   # Пряме присвоєння — ніхто не зупиняє!

print(f"mpg після злому:       {car._mpg}")
print(f"efficiency після злому: {car.efficiency()}")
print()
print("Від'ємна ефективність — це не edge-case.")
print("Це ВІДСУТНІСТЬ КОНТРОЛЮ над станом об'єкта.")

> ### 💥 Що ми щойно побачили?
>
> `set_mpg()` виконує свою роботу, але її **легко проігнорувати**.  
> Об'єкт залишається вразливим через `_mpg`.
>
> **Запитання:**  
> Чи можемо ми змусити Python автоматично викликати валідацію —  
> при **будь-якому** записі, навіть через `car.mpg = ...`?

---

> ### Так. Це називається `@property`.

---

## ✅ Розділ 3 — Рішення 1: `@property`

`@property` дозволяє **замінити `set_mpg()` на автоматичну перевірку**.  
Зовні виглядає як звичайний атрибут — але зсередини це **контрольований доступ**.

In [ ]:
class Car:
    def __init__(self, name, mpg, weight, origin):
        self.name    = name
        self.mpg     = mpg     # Виклик setter-а прямо в __init__!
        self._weight = weight
        self.origin  = origin

    # --- @property: геттер ---
    @property
    def mpg(self):
        return self._mpg

    # --- @mpg.setter: валідація при будь-якому записі ---
    @mpg.setter
    def mpg(self, value):
        if value <= 0:
            raise ValueError(f"mpg має бути > 0, отримано {value}")
        self._mpg = value

    def efficiency(self):
        return round(self._mpg, 2)

    def __str__(self):
        return f"{self.name} — {self._mpg} mpg [{self.origin}]"

    def __lt__(self, other):
        return self._mpg < other._mpg

    def __eq__(self, other):
        if not isinstance(other, Car): return NotImplemented
        return self._mpg == other._mpg

In [ ]:
# --- Тест 1: правильне значення ---
car = Car("Ford Pinto", 18.0, 2601, "usa")
print(car)

# --- Тест 2: спроба злому через пряме присвоєння ---
try:
    car.mpg = -100    # Python автоматично викликає setter!
except ValueError as e:
    print(f"ValueError: {e}")

print(f"mpg залишився незмінним: {car.mpg}")

# --- Тест 3: правильне оновлення ---
car.mpg = 25.0
print(f"mpg оновлено: {car.mpg}")

### 🔥 Інсайт

```python
car.mpg = -100   # ← звичайне присвоєння
```

**Python автоматично викликав `@mpg.setter`.**  
Ти не викликаєш `set_mpg()`. Ти не думаєш про захист.  
**Система захищає себе сама.**

> `@property` перетворює метод на контрольований атрибут.  
> Зовні — `car.mpg`. Зсередини — `__get__` і `__set__`.

---

Але у `@property` є одна серйозна проблема.

---

## ❌ Розділ 4 — Обмеження: `@property` не масштабується

Реальний `Car` має кілька числових полів, кожне з яких потребує валідації:

| Поле | Правило |
|------|--------|
| `mpg` | > 0 |
| `weight` | > 0 |
| `horsepower` | > 0 |

Подивимось, що станеться, якщо додати `weight` і `horsepower` через `@property`:

In [ ]:
# Що ДОВЕДЕТЬСЯ НАПИСАТИ, якщо захищати кожне поле через @property:

class CarWithManyProperties:
    def __init__(self, name, mpg, weight, horsepower, origin):
        self.name        = name
        self.mpg         = mpg
        self.weight      = weight
        self.horsepower  = horsepower
        self.origin      = origin

    # ---- mpg ----
    @property
    def mpg(self):
        return self._mpg

    @mpg.setter
    def mpg(self, value):
        if value <= 0:                          # <-- та сама перевірка
            raise ValueError(f"mpg має бути > 0, отримано {value}")
        self._mpg = value

    # ---- weight ----
    @property
    def weight(self):
        return self._weight

    @weight.setter
    def weight(self, value):
        if value <= 0:                          # <-- та сама перевірка знову!
            raise ValueError(f"weight має бути > 0, отримано {value}")
        self._weight = value

    # ---- horsepower ----
    @property
    def horsepower(self):
        return self._horsepower

    @horsepower.setter
    def horsepower(self, value):
        if value <= 0:                          # <-- та сама перевірка ВТРЕТЄ!
            raise ValueError(f"horsepower має бути > 0, отримано {value}")
        self._horsepower = value

print("Клас написано. Але чи відчуваєш ти біль?")
print("Одна й та сама логіка — тричі.")
print("А якщо полів стане 10? Або 50?")

> ### 💥 Проблема дублювання
>
> Ми тричі написали одну й ту саму логіку:
> ```python
> if value <= 0:
>     raise ValueError(...)
> ```
>
> Якщо правило зміниться (наприклад, `>= 1` замість `> 0`) —  
> доведеться правити **у трьох місцях**.  
> Забудеш одне — отримаєш баг, якого важко знайти.
>
> **Повторення — ознака незавершеної абстракції.**  
> Нам потрібен інструмент, який ОДИН РАЗ описує правило і застосовує його скрізь.

---

## ✅ Розділ 5 — Рішення 2: Дескриптор `PositiveNumber`

**Ключова ідея:**  
Замість того, щоб писати логіку валідації **всередині кожного класу**,  
ми виносимо її в **окремий об'єкт-охоронець** — дескриптор.

Дескриптор — це клас з методами `__get__` і `__set__`.  
Коли ти прив'язуєш його як **атрибут класу**, він автоматично перехоплює  
будь-який читання і запис до цього атрибута.

> Думай про нього як про **розумний замок на дверях**:  
> ти встановлюєш його один раз — він охороняє назавжди.

In [ ]:
class PositiveNumber:
    # Дескриптор: охороняє будь-яке числове поле, яке має бути > 0

    def __set_name__(self, owner, name):
        # Python викликає це автоматично при визначенні класу
        # name — ім'я атрибута (mpg, weight, horsepower)
        self.name    = name
        self.storage = "_" + name  # де насправді зберігаємо дані

    def __get__(self, instance, owner):
        if instance is None:
            # Доступ через клас (Car.mpg) — повертаємо сам дескриптор
            return self
        # Читаємо значення зі словника екземпляра
        return getattr(instance, self.storage)

    def __set__(self, instance, value):
        # Ця логіка виконується при КОЖНОМУ записі: car.mpg = ...
        if not isinstance(value, (int, float)):
            raise TypeError(f"{self.name} має бути числом, отримано {type(value).__name__}")
        if value <= 0:
            raise ValueError(f"{self.name} має бути > 0, отримано {value}")
        # Зберігаємо у словник екземпляра (НЕ у дескриптор!)
        setattr(instance, self.storage, value)


print("Дескриптор PositiveNumber визначено.")
print("Один клас — одне правило — необмежена кількість полів.")

---

## Розділ 6 — Застосовуємо дескриптор до `Car`

Тепер ми прив'язуємо `PositiveNumber` як **атрибут класу**.  
Одна рядок — і все поле захищено назавжди.

In [ ]:
class Car:
    # Дескриптори оголошуємо на рівні класу — один раз!
    mpg        = PositiveNumber()
    weight     = PositiveNumber()
    horsepower = PositiveNumber()

    def __init__(self, name, mpg, weight, horsepower, origin):
        self.name       = name
        self.mpg        = mpg           # -> PositiveNumber.__set__
        self.weight     = weight        # -> PositiveNumber.__set__
        self.horsepower = horsepower    # -> PositiveNumber.__set__
        self.origin     = origin

    def efficiency(self):
        return round(self.mpg, 2)

    def power_to_weight(self):
        # Відношення потужності до ваги — показник "спортивності"
        return round(self.horsepower / self.weight * 1000, 2)

    def __str__(self):
        return f"{self.name} — {self.mpg} mpg, {self.horsepower} к.с. [{self.origin}]"

    def __repr__(self):
        return (f"Car({self.name!r}, mpg={self.mpg}, "
                f"weight={self.weight}, hp={self.horsepower})")

    def __lt__(self, other):
        return self.mpg < other.mpg

    def __eq__(self, other):
        if not isinstance(other, Car): return NotImplemented
        return self.mpg == other.mpg


class GasolineCar(Car):
    def efficiency(self): return round(self.mpg * 1.0, 2)

class DieselCar(Car):
    def efficiency(self): return round(self.mpg * 1.2, 2)

class ElectricCar(Car):
    def efficiency(self): return round(self.mpg * 2.5, 2)


print("Клас Car з дескрипторами готовий.")

In [ ]:
# --- Тест 1: створення валідних об'єктів ---
cars = [
    GasolineCar("Ford Pinto",    18.0, 2601, 75,  "usa"),
    DieselCar(  "VW Rabbit",     30.0, 1925, 90,  "europe"),
    ElectricCar("Tesla Model S", 100.0, 4647, 670, "usa"),
]

for car in cars:
    print(car)

In [ ]:
# --- Тест 2:  ---
# Спробуємо присвоїти невалідні значення напряму

car = GasolineCar("Ford Pinto", 18.0, 2601, 75, "usa")

print("Спроба 1: car.mpg = -100")
try:
    car.mpg = -100
except ValueError as e:
    print(f"  ValueError: {e}")

print()
print("Спроба 2: car.weight = 0")
try:
    car.weight = 0
except ValueError as e:
    print(f"  ValueError: {e}")

print()
print("Спроба 3: car.horsepower = -50")
try:
    car.horsepower = -50
except ValueError as e:
    print(f"  ValueError: {e}")

print()
print(f"Стан car після всіх спроб: mpg={car.mpg}, weight={car.weight}, hp={car.horsepower}")
print("Жодне значення не змінилось. Система захистила себе.")

### 🔥 Інсайт

```python
car.mpg = -100      # ValueError — дескриптор перехопив
car.weight = 0      # ValueError — той самий дескриптор
car.horsepower = -50 # ValueError — знову він
```

**Ми написали правило валідації ОДИН РАЗ** у класі `PositiveNumber`.  
Він охороняє `mpg`, `weight`, `horsepower` — і будь-яке інше поле, яке ми додамо.

> **Тепер ти не перевіряєш дані.  
> Система перевіряє їх за тебе.**

---

## Розділ 6.1 — Реальний датасет: система, що захищає себе

Тепер побудуємо парк автомобілів з реального датасету `mpg`.  
Кожен об'єкт — автоматично валідований.

*(Датасет не має `horsepower` для всіх рядків — пропускаємо `NaN`)*

In [ ]:
print(df.info())
print(df.describe())
print(df.describe(include="all"))

In [ ]:
# Будуємо реальний автопарк
df = df.copy()

for col in ["mpg", "weight", "horsepower"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["mpg", "weight", "horsepower"])
real_cars = []
for _, row in df.iterrows():
    # Пропускаємо рядки з відсутнім horsepower
    if pd.isna(row["horsepower"]):
        continue
    try:
        car = GasolineCar(
            name       = row["name"],
            mpg        = row["mpg"],
            weight     = row["weight"],
            horsepower = row["horsepower"],
            origin     = row["origin"]
        )
        real_cars.append(car)
    except (ValueError, TypeError) as e:
        print("❌", e, "|", row["name"])

print(f"Валідних автомобілів у парку: {len(real_cars)}")
print()

# Топ-5 за mpg
top5 = sorted(real_cars)[-5:]
print(f"Топ-5 за паливною ефективністю:")
for car in reversed(top5):
    print(f"  {car}")

---

## Розділ 7 — Під капотом: `@property` і дескриптор — одне й те саме

Важливе розуміння:

> **`@property` — це НЕ особливий синтаксис Python.**  
> Це просто вбудований клас-дескриптор.

Коли ти пишеш:

```python
@property
def mpg(self):
    return self._mpg
```

Python робить рівно те саме, що й:

```python
mpg = property(fget=lambda self: self._mpg)
```

А `property` — це клас з методами `__get__`, `__set__`, `__delete__`.  
Тобто `property` — **це дескриптор**, який Python написав за нас у C.

In [ ]:
# Покажемо що property == дескриптор зсередини

# Спрощена Python-реалізація вбудованого property:
class MyProperty:

    def __init__(self, fget=None, fset=None):
        self.fget = fget
        self.fset = fset

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return self.fget(instance)   # Виклик геттера

    def __set__(self, instance, value):
        if self.fset is None:
            raise AttributeError("can't set attribute")
        self.fset(instance, value)   # Виклик сеттера

    def setter(self, fset):
        return MyProperty(self.fget, fset)


# Використовуємо нашу MyProperty замість вбудованої property
class SimpleCar:
    def __init__(self, mpg):
        self.mpg = mpg

    def _get_mpg(self):
        return self._mpg

    def _set_mpg(self, value):
        if value <= 0:
            raise ValueError(f"mpg > 0, отримано {value}")
        self._mpg = value

    # MyProperty — наш аналог @property
    mpg = MyProperty(_get_mpg, _set_mpg)


sc = SimpleCar(18.0)
print(f"mpg = {sc.mpg}")

try:
    sc.mpg = -5
except ValueError as e:
    print(f"ValueError: {e}")

print()
print("MyProperty поводиться точно як @property.")
print("Тому що @property — це саме такий же дескриптор, тільки написаний на C.")

### Ключовий висновок

| | `@property` | Дескриптор (`PositiveNumber`) |
|---|---|---|
| **Що це** | Вбудований дескриптор | Твій власний дескриптор |
| **Де живе логіка** | Прямо у класі | В окремому класі-охоронці |
| **Повторюваність** | Кожне поле — окремий код | Один клас — всі поля |
| **Рівень** | Middle | Senior |

---

**Аналогія:**  
`@property` — це **один унікальний замок**, зроблений вручну для конкретних дверей.  
Дескриптор — це **завод з виробництва замків**: пишеш один раз, ставиш скрізь.

---

## Розділ 8 — Порівняльна таблиця підходів

| Підхід | Контроль | Повторне використання | Pythonic? | Рівень |
|--------|----------|-----------------------|-----------|--------|
| Прямий доступ `_mpg = value` | ❌ Ніякого | — | ❌ | Початківець |
| Методи `set_mpg()` / `get_mpg()` | ✅ Ручний | ❌ Низьке | ❌ (Java-стиль) | Junior |
| `@property` | ✅ Автоматичний | ⚠️ Середнє (дублювання) | ✅ | Middle |
| Дескриптор | ✅ Автоматичний | ✅ Максимальне | ✅ | Senior |

---

### Коли що використовувати?

```
1 поле з валідацією      ->  @property
2+ поля з однаковим правилом  ->  Дескриптор
Без валідації            ->  Простий публічний атрибут
```

> **Правило Python-архітектора:**  
> Починай з простого (`self.mpg = mpg`).  
> Якщо потрібна валідація одного поля — `@property`.  
> Якщо та сама логіка повторюється 2+ рази — дескриптор.

---

## Розділ 9 — Завдання

### 🔒 Завдання 1 — `@property` для `acceleration`

Додай до класу `Car` захищений атрибут `acceleration`:  
- Правило: `0 < acceleration <= 30` (реальний діапазон 0-60 mph у секундах)  
- Використай `@property` та `@acceleration.setter`  
- При порушенні — `ValueError`

In [ ]:
# TODO: Завдання 1
# Додай @property acceleration до Car

class CarWithAcceleration(Car):
    def __init__(self, name, mpg, weight, horsepower, acceleration, origin):
        super().__init__(name, mpg, weight, horsepower, origin)
        self.acceleration = acceleration   # буде викликати твій setter

    # TODO: додай @property та @acceleration.setter тут
    # Правило: 0 < acceleration <= 30
    pass


# Тест:
# car = CarWithAcceleration("Toyota", 30.0, 2200, 90, 12.5, "japan")
# print(car.acceleration)  # 12.5
# car.acceleration = -1    # ValueError
# car.acceleration = 100   # ValueError

### 🔥 Завдання 2 — Дескриптор `RangedNumber`

Напиши дескриптор `RangedNumber(min_val, max_val)` —  
перевіряє, що значення потрапляє у заданий діапазон.

Потім застосуй його до класу `Car`:
```python
class Car:
    mpg        = RangedNumber(1, 200)
    weight     = RangedNumber(500, 10000)
    horsepower = RangedNumber(10, 2000)
```

In [ ]:
# TODO: Завдання 2

class RangedNumber:
    def __init__(self, min_val, max_val):
        self.min_val = min_val
        self.max_val = max_val

    def __set_name__(self, owner, name):
        pass  # TODO: зберегти ім'я

    def __get__(self, instance, owner):
        pass  # TODO

    def __set__(self, instance, value):
        pass  # TODO: перевірити min_val <= value <= max_val


# Після реалізації:
# class StrictCar:
#     mpg        = RangedNumber(1, 200)
#     weight     = RangedNumber(500, 10000)
#     horsepower = RangedNumber(10, 2000)
#
#     def __init__(self, name, mpg, weight, horsepower, origin):
#         self.name = name
#         self.mpg = mpg
#         self.weight = weight
#         self.horsepower = horsepower
#         self.origin = origin
#
# sc = StrictCar("VW", 30, 1925, 90, "europe")   # OK
# sc.weight = 100   # ValueError: поза діапазоном

### ✨ Завдання 3 — Дескриптор для `origin`

Напиши дескриптор `ValidatedChoice(choices)`, що дозволяє лише значення зі списку.  
Застосуй до `Car.origin`:

```python
class Car:
    origin = ValidatedChoice(["usa", "europe", "japan"])
```

Спроба `car.origin = "mars"` — `ValueError`.

In [ ]:
# TODO: Завдання 3

class ValidatedChoice:
    def __init__(self, choices):
        self.choices = choices

    def __set_name__(self, owner, name):
        pass  # TODO

    def __get__(self, instance, owner):
        pass  # TODO

    def __set__(self, instance, value):
        pass  # TODO: перевірити що value в self.choices


# Після реалізації:
# class ValidatedCar:
#     origin = ValidatedChoice(["usa", "europe", "japan"])
#
#     def __init__(self, name, origin):
#         self.name   = name
#         self.origin = origin
#
# vc = ValidatedCar("Ford", "usa")    # OK
# vc.origin = "mars"                  # ValueError!

### 🧠 Завдання 4 — Фінальна система

Збери фінальний клас `Car` з усіма дескрипторами:  
`mpg = PositiveNumber()`, `weight = PositiveNumber()`, `horsepower = PositiveNumber()`, `origin = ValidatedChoice(...)`

Потім:  
1. Завантаж реальні дані з `df`  
2. Порахуй, скільки рядків пройдуть валідацію  
3. Знайди найефективніший автомобіль кожної країни

In [ ]:
# TODO: Завдання 4

# class FinalCar:
#     mpg        = PositiveNumber()
#     weight     = PositiveNumber()
#     horsepower = PositiveNumber()
#     origin     = ValidatedChoice(["usa", "europe", "japan"])
#
#     ...

# valid_cars = []
# for _, row in df.iterrows():
#     try:
#         valid_cars.append(FinalCar(...))
#     except (ValueError, TypeError):
#         pass
#
# print(f"Валідних: {len(valid_cars)} з {len(df)}")

---

## Розділ 10 — Питання для інтерв'ю

> Це не просто домашка — це підготовка до реального технічного інтерв'ю.

---

### ❓ Питання 1: Чим `@property` відрізняється від методів `get/set`?

**✅ Відповідь:**

| | `set_mpg()` / `get_mpg()` | `@property` |
|---|---|---|
| Синтаксис виклику | `car.set_mpg(25)` | `car.mpg = 25` |
| Стиль | Java-стиль | Python (Pythonic) |
| Обхід захисту | `car._mpg = -100` все ще можливий | `car.mpg = -100` — перехоплено |
| Явність | Явний метод | Неявний виклик getter/setter |

---

### ❓ Питання 2: Що таке дескриптор?

**✅ Відповідь:**

Дескриптор — це клас з методами `__get__` та/або `__set__`, що керує доступом до атрибута **іншого** класу.  
Коли об'єкт-дескриптор оголошується як **атрибут класу**, Python автоматично викликає його методи  
при кожному читанні або записі цього атрибута.

```python
class Car:
    mpg = PositiveNumber()   # дескриптор-атрибут класу
```

---

### ❓ Питання 3: Чим відрізняється data descriptor від non-data descriptor?

**✅ Відповідь:**

| | Data Descriptor | Non-data Descriptor |
|---|---|---|
| Реалізує | `__get__` + `__set__` | Тільки `__get__` |
| Пріоритет | Вищий, ніж `instance.__dict__` | Нижчий |
| Можна перекрити? | Ні | Так (через `obj.attr = ...`) |
| Приклади | `property`, `PositiveNumber` | Звичайні методи класу |

---

## Розділ 11 — Висновок: Еволюція контролю

### Що ми збудували

| Урок | Підхід | Механізм |
|------|--------|----------|
| **17** | Ручний контроль | `set_mpg()` — але легко обійти |
| **18** | Автоматичний контроль | `@property` — перехоплює один атрибут |
| **18** | Масштабований контроль | Дескриптор — перехоплює будь-яку кількість атрибутів |

---

> ### 💡 Головне повідомлення уроку
>
> **У Уроці 17 ти перевіряв дані вручну.**  
> **Сьогодні ти навчив систему перевіряти їх за тебе.**
>
> `@property` — Python перехоплює один атрибут.  
> Дескриптор — Python перехоплює скільки завгодно атрибутів.  
>
> **Тепер ти не перевіряєш дані.  
> Система перевіряє їх за тебе.**

---

> *«ООП — це контроль реальності через код.»*  
> У цьому уроці ми підняли цей контроль на новий рівень.